# Avviare il Progetto Diffusion su Google Colab (via GitHub)
Questo notebook scarica automaticamente il codice da GitHub, monta Google Drive per il salvataggio dei pesi e lancia l'addestramento della rete Diffusion.

### 1. Scarica / Aggiorna il codice da GitHub

In [ ]:
import os

repo_url = "https://github.com/giorgio-di-dio/vessel-project.git"
repo_name = "vessel-project"

if not os.path.exists(repo_name):
    print(f"Clonazione della repository...")
    !git clone -b main {repo_url}
else:
    print("Repository già presente.")

# Spostati nella cartella del progetto
%cd {repo_name}

# Aggiorna all'ultima versione
!git fetch origin
!git pull origin main

### 2. Installa le librerie necessarie

In [ ]:
!pip install -r requirements.txt

### 3. Monta Google Drive per il salvataggio persistente

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 4. Setup della Directory di Salvataggio su Drive

In [ ]:
import datetime
from zoneinfo import ZoneInfo
import os

# Genera il timestamp per questa run di Diffusion
run_timestamp = datetime.datetime.now(tz=ZoneInfo('Europe/Rome')).strftime("%Y%m%d_%H%M")
DRIVE_SAVE_DIR = f"/content/drive/MyDrive/Advanced_Machine_Learning/output_pesi/diffusion_{run_timestamp}"

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"I checkpoint di questa run verranno salvati in: {DRIVE_SAVE_DIR}")

### 5. Inizializzazione Dati e Rete Diffusion

In [ ]:
import sys
import os
import torch

# 1. Aggiungiamo forzatamente la cartella del progetto al path di Python
# Questo risolve in modo definitivo l'errore di importazione su Colab!
project_path = '/content/vessel-project'
if project_path not in sys.path:
    sys.path.append(project_path)

# Ora le importazioni funzioneranno perfettamente
from src.config import get_kaggle_dataset_path
from src.dataset import get_dataloaders
from src.models.diffusion_scheduler import DiffusionScheduler
from src.models.diffusion_unet import ConditionalUNet
from src.engine_diffusion import train_diffusion

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device in uso: {DEVICE}")

# Pesca il dataset Kaggle
dataset_root = get_kaggle_dataset_path()

# Genera il dataloader
train_loader, val_loader, _ = get_dataloaders(
    dataset_root=dataset_root,
    patch_size=256, 
    batch_size=8,
    num_workers=2
)

# Inizializza Scheduler e Modello
scheduler = DiffusionScheduler(num_timesteps=1000, device=DEVICE)
model = ConditionalUNet(image_channels=3, mask_channels=1, base_dim=32).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

### 6. Avvia l'Addestramento!

In [ ]:
local_save_dir = "output/models/diffusion_checkpoints"
os.makedirs(local_save_dir, exist_ok=True)

train_diffusion(
    model=model,
    dataloader=train_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=50,
    device=DEVICE,
    save_dir=local_save_dir,
    drive_save_dir=DRIVE_SAVE_DIR
)

print(f"Training completato! Pesi salvati in locale e in {DRIVE_SAVE_DIR}")

In [ ]:
# Scollega il runtime a fine esecuzione
#from google.colab import runtime
#runtime.unassign()